In [1]:
import pandas as pd

# Carrega os CSVs da execução com 10 repetições: 12 métodos (run_parallel.py)
# + 5 variantes do SMQ (run_smq_complementar.py), pareados entre si
CSV_PATHS = ["results_few_shot_quantification_10reps.csv", "results_smq_10reps.csv"]
df = pd.concat([pd.read_csv(p) for p in CSV_PATHS], ignore_index=True)
print(f"Repetições por dataset: {df.groupby('dataset')['repeticao'].nunique().unique()}")
print(f"Métodos ({df['metodo'].nunique()}): {sorted(df['metodo'].unique())}")

# 1. Quantidade total de datasets ÚNICOS
total_datasets = df['dataset'].nunique()
print(f"Total de datasets únicos: {total_datasets}")

# 2. Lista com os nomes de cada dataset
nomes_datasets = df['dataset'].unique()
print("Nomes dos datasets:", nomes_datasets)

# 3. Contagem de repetições/experimentos por dataset
contagem_por_dataset = df['dataset'].value_counts()
print(contagem_por_dataset)

Repetições por dataset: [10]
Métodos (17): ['ACC', 'CC', 'DyS', 'EMQ', 'FSGMM', 'FSGPEM', 'FSMM', 'FSNN', 'FSPL', 'KDEyML', 'PACC', 'PCC', 'smq_base', 'smq_et', 'smq_gbr', 'smq_iso', 'smq_rf']
Total de datasets únicos: 30
Nomes dos datasets: <StringArray>
[ 'pageblocks.5',  'wine-q-white',      'spambase',       'semeion',
         'ctg.1',         'ctg.2',         'ctg.3',        'german',
          'wdbc',    'wine-q-red',         'cmc.2',         'cmc.1',
         'cmc.3',         'sonar',    'ionosphere',        'spectf',
         'yeast', 'breast-cancer',     'tictactoe',  'mammographic',
   'transfusion',     'balance.1',        'wine.1',     'balance.3',
        'wine.2',        'wine.3',        'iris.1',      'haberman',
        'iris.2',        'iris.3']
Length: 30, dtype: str
dataset
pageblocks.5     1140
wine-q-white     1140
spambase         1140
semeion          1140
ctg.1            1140
ctg.2            1140
ctg.3            1140
german           1140
wdbc             11

In [2]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import seaborn as sns
import os
import shutil

# ============================================================
# CONFIG
# ============================================================

# Execução com 10 repetições: 12 métodos (run_parallel.py) + 5 variantes do
# SMQ (run_smq_complementar.py), pareados entre si
CSV_PATHS = [
    "results_few_shot_quantification_10reps.csv",
    "results_smq_10reps.csv",
]
OS_DIR = "boxplots_binario"

# Métrica utilizada para o ranking. Os CSVs trazem uma linha por
# (dataset, repetição, n, método) já resumida sobre os batches de teste,
# então a métrica é a média do AE nos batches (ae_mean).
METRIC = "ae_mean"

# Limpa a pasta de saída
shutil.rmtree(OS_DIR, ignore_errors=True)
os.makedirs(OS_DIR, exist_ok=True)


# ============================================================
# PUBLICATION STYLE
# ============================================================

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "Times", "DejaVu Serif"],
    "font.size": 10,
    "axes.labelsize": 10,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "axes.linewidth": 0.4,

    "xtick.major.width": 0.0,
    "ytick.major.width": 0.0,
    "xtick.minor.width": 0.0,
    "ytick.minor.width": 0.0,

    "xtick.major.size": 0.0,
    "ytick.major.size": 0.0,
    "xtick.minor.size": 0.0,
    "ytick.minor.size": 0.0,

    "xtick.direction": "in",
    "ytick.direction": "in",

    "xtick.top": False,
    "ytick.right": False,

    "axes.grid": True,
    "axes.grid.axis": "y",

    "grid.linewidth": 0.4,
    "grid.alpha": 0.5,
    "grid.color": "#aaaaaa",

    "figure.dpi": 300,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
    "savefig.pad_inches": 0.05,
})


# ============================================================
# LOAD CSV
# ============================================================

print(f"Carregando {CSV_PATHS}...")

df_master = pd.concat(
    [pd.read_csv(p) for p in CSV_PATHS],
    ignore_index=True
)

print(f"Linhas carregadas: {len(df_master):,}")
print(f"Colunas: {df_master.columns.tolist()}")


# ============================================================
# VERIFICAÇÃO DAS COLUNAS
# ============================================================

required_columns = {"dataset", "metodo", "n", METRIC}

missing_columns = required_columns - set(df_master.columns)

if missing_columns:
    raise ValueError(
        f"Colunas ausentes: {missing_columns}\n"
        f"Colunas disponíveis: {df_master.columns.tolist()}"
    )


# ============================================================
# LIMPEZA DOS DADOS
# ============================================================

df_master = df_master.copy()

df_master[METRIC] = pd.to_numeric(
    df_master[METRIC],
    errors="coerce"
)

df_master = df_master.dropna(
    subset=["dataset", "metodo", "n", METRIC]
)


# ============================================================
# PADRONIZA NOMES DOS MÉTODOS
# ============================================================

df_master["modelo"] = df_master["metodo"].replace({

    # SMQ
    "smq_base": "SMQ_BASE",
    "smq_rf": "SMQ_RF",
    "smq_et": "SMQ_ET",
    "smq_gbr": "SMQ_GBR",
    "smq_iso": "SMQ_ISO",

    # Baselines
    "CC": "CC",
    "PCC": "PCC",
    "ACC": "ACC",
    "PACC": "PACC",
    "EMQ": "EMQ",
    "DyS": "DyS",
    "KDEyML": "KDEyML",

    # Few-shot
    "FSNN": "FSNN",
    "FSMM": "FSMM",
    "FSGMM": "FSGMM",
    "FSGPEM": "FSGPEM",
    "FSPL": "FSPL",
})


# ============================================================
# LABELS DOS MÉTODOS
# ============================================================

label_map = {

    # SMQ
    "SMQ_BASE": r"$\mathrm{SMQ}$",
    "SMQ_ISO": r"$\mathrm{SMQ}_{\mathrm{ISO}}$",
    "SMQ_RF": r"$\mathrm{SMQ}_{\mathrm{RF}}$",
    "SMQ_ET": r"$\mathrm{SMQ}_{\mathrm{ET}}$",
    "SMQ_GBR": r"$\mathrm{SMQ}_{\mathrm{GBR}}$",

    # Baselines
    "CC": r"$\mathrm{CC}$",
    "PCC": r"$\mathrm{PCC}$",
    "ACC": r"$\mathrm{ACC}$",
    "PACC": r"$\mathrm{PACC}$",
    "EMQ": r"$\mathrm{EMQ}$",
    "DyS": r"$\mathrm{DyS}$",
    "KDEyML": r"$\mathrm{KDEy}_{\mathrm{ML}}$",

    # Few-shot
    "FSNN": r"$\mathrm{FSNN}$",
    "FSMM": r"$\mathrm{FSMM}$",
    "FSGMM": r"$\mathrm{FSGMM}$",
    "FSGPEM": r"$\mathrm{FSGPEM}$",
    "FSPL": r"$\mathrm{FSPL}$",
}


# ============================================================
# ORDENA OS VALORES DE N
# ============================================================

# Converte tudo para string para tratar "full"
df_master["n_str"] = df_master["n"].astype(str)

# Ordem desejada dos gráficos
n_order_preferred = ["1", "2", "3", "5", "10", "100", "full"]

# Pega apenas valores que realmente existem no CSV
unique_n = df_master["n_str"].unique().tolist()

n_values = [
    n for n in n_order_preferred
    if n in unique_n
]

# Caso apareça algum valor inesperado no CSV
remaining_n = [
    n for n in unique_n
    if n not in n_values
]

n_values.extend(
    sorted(
        remaining_n,
        key=lambda x: (
            x == "full",
            float(x) if x.replace(".", "", 1).isdigit() else float("inf")
        )
    )
)


# ============================================================
# INFORMAÇÕES
# ============================================================

print("\n========================================")
print("CONFIGURAÇÃO")
print("========================================")

print(f"Métrica: {METRIC.upper()}")
print(f"Valores de n encontrados: {n_values}")
print(f"Datasets: {df_master['dataset'].nunique()}")
print(f"Métodos: {df_master['modelo'].nunique()}")

print("========================================\n")


# ============================================================
# LOOP: UM GRÁFICO PARA CADA N
# ============================================================

for n_value in n_values:

    print(f"Gerando gráfico para n = {n_value}...")

    # --------------------------------------------------------
    # FILTRA APENAS O N ATUAL
    # --------------------------------------------------------

    df_n = df_master[
        df_master["n_str"] == str(n_value)
    ].copy()

    # --------------------------------------------------------
    # MÉDIA DO AE
    #
    # Média considerando:
    #
    # dataset + modelo
    #
    # Aqui são agregados batches e repetições,
    # mas APENAS para o n atual.
    # --------------------------------------------------------

    dataset_mean = (
        df_n
        .groupby(["dataset", "modelo"])[METRIC]
        .mean()
        .reset_index()
    )

    # --------------------------------------------------------
    # RANK POR DATASET
    #
    # Menor AE = melhor = Rank 1
    # --------------------------------------------------------

    dataset_mean["rank_dataset"] = (
        dataset_mean
        .groupby("dataset")[METRIC]
        .rank(
            method="average",
            ascending=True
        )
    )

    # --------------------------------------------------------
    # ORDENA MÉTODOS
    #
    # Melhor → pior
    #
    # Critério:
    # menor mediana do ranking
    # --------------------------------------------------------

    ordem_rank = (
        dataset_mean
        .groupby("modelo")["rank_dataset"]
        .median()
        .sort_values()
        .index
        .tolist()
    )

    n_models = len(ordem_rank)

    # Labels formatados
    display_labels = [
        label_map.get(modelo, modelo)
        for modelo in ordem_rank
    ]

    # --------------------------------------------------------
    # PALETA
    # --------------------------------------------------------

    palette = sns.color_palette(
        "Spectral",
        n_colors=n_models
    )

    # --------------------------------------------------------
    # TAMANHO DO GRÁFICO
    # --------------------------------------------------------

    fig_width = max(
        7.2,
        n_models * 0.55
    )

    fig, ax = plt.subplots(
        figsize=(fig_width, 3.5)
    )

    # --------------------------------------------------------
    # BOX PLOT
    # --------------------------------------------------------

    sns.boxplot(
        data=dataset_mean,

        x="modelo",
        y="rank_dataset",

        hue="modelo",

        order=ordem_rank,
        hue_order=ordem_rank,

        palette=palette,

        dodge=False,

        legend=False,

        width=0.55,

        linewidth=0.8,

        flierprops=dict(
            marker="o",
            markerfacecolor="none",
            markeredgecolor="#555555",
            markeredgewidth=0.6,
            markersize=3.5,
        ),

        medianprops=dict(
            color="black",
            linewidth=1.4
        ),

        whiskerprops=dict(
            linewidth=0.8
        ),

        capprops=dict(
            linewidth=0.8
        ),

        boxprops=dict(
            linewidth=0.8
        ),

        ax=ax,
    )

    # --------------------------------------------------------
    # TÍTULO
    # --------------------------------------------------------

    if str(n_value).lower() == "full":

        title_n = "Full Training Set"

    else:

        title_n = f"n = {n_value} per class"

    ax.set_title(
        title_n,
        fontsize=11,
        pad=10
    )

    # --------------------------------------------------------
    # AXES
    # --------------------------------------------------------

    ax.set_xlabel(
        "Methods",
        labelpad=6
    )

    ax.set_ylabel(
        "Rank (AE)",
        labelpad=6
    )

    # --------------------------------------------------------
    # LIMITE Y
    # --------------------------------------------------------

    ax.set_ylim(
        0,
        n_models + 1
    )

    ax.yaxis.set_major_locator(
        ticker.MultipleLocator(2)
    )

    ax.yaxis.set_minor_locator(
        ticker.NullLocator()
    )

    # --------------------------------------------------------
    # SPINES
    # --------------------------------------------------------

    for spine in ax.spines.values():

        spine.set_linewidth(0.4)

        spine.set_color(
            "#aaaaaa"
        )

    # --------------------------------------------------------
    # TICKS
    # --------------------------------------------------------

    ax.tick_params(
        which="both",
        length=0
    )

    ax.set_xticks(
        range(n_models)
    )

    ax.set_xticklabels(
        display_labels,

        rotation=40,

        ha="right",

        rotation_mode="anchor"
    )

    # --------------------------------------------------------
    # LAYOUT
    # --------------------------------------------------------

    plt.tight_layout()

    # --------------------------------------------------------
    # SAVE
    # --------------------------------------------------------

    safe_n_value = str(n_value).replace("/", "_")

    nome_arquivo = (
        f"rank_boxplot_binario_n_{safe_n_value}.png"
    )

    output_path = os.path.join(
        OS_DIR,
        nome_arquivo
    )

    plt.savefig(
        output_path,
        dpi=300
    )

    plt.close(fig)

    print(
        f"  Salvo: {output_path}"
    )

    # --------------------------------------------------------
    # PRINT RANKING
    # --------------------------------------------------------

    print(
        f"\nRanking para n = {n_value}"
    )

    for i, modelo in enumerate(
        ordem_rank,
        start=1
    ):

        median_rank = (
            dataset_mean[
                dataset_mean["modelo"] == modelo
            ]["rank_dataset"]
            .median()
        )

        mean_ae = (
            dataset_mean[
                dataset_mean["modelo"] == modelo
            ][METRIC]
            .mean()
        )

        print(
            f"{i:2d}. "
            f"{modelo:<10} "
            f"| Median Rank = {median_rank:.2f} "
            f"| Mean AE = {mean_ae:.6f}"
        )

    print()


# ============================================================
# FINAL
# ============================================================

print("========================================")
print("CONCLUÍDO!")
print("========================================")

print(f"Todos os gráficos foram salvos em:")
print(OS_DIR)

print("\nGráficos gerados:")

for filename in sorted(os.listdir(OS_DIR)):
    print(f" - {filename}")

Carregando ['results_few_shot_quantification_10reps.csv', 'results_smq_10reps.csv']...
Linhas carregadas: 34,200
Colunas: ['dataset', 'n', 'metodo', 'ae_mean', 'ae_std', 'ae_median', 'rae_mean', 'rae_std', 'rae_median', 'n_batches', 'repeticao', 'pareado']

CONFIGURAÇÃO
Métrica: AE_MEAN
Valores de n encontrados: ['1', '2', '3', '5', '10', '100', 'full']
Datasets: 30
Métodos: 17

Gerando gráfico para n = 1...


  Salvo: boxplots_binario\rank_boxplot_binario_n_1.png

Ranking para n = 1
 1. FSMM       | Median Rank = 1.00 | Mean AE = 0.216534
 2. PCC        | Median Rank = 3.50 | Mean AE = 0.244210
 3. SMQ_ET     | Median Rank = 4.00 | Mean AE = 0.244294
 4. FSGMM      | Median Rank = 4.50 | Mean AE = 0.247113
 5. CC         | Median Rank = 5.50 | Mean AE = 0.253049
 6. FSNN       | Median Rank = 6.50 | Mean AE = 0.252803
 7. SMQ_GBR    | Median Rank = 7.00 | Mean AE = 0.250900
 8. FSGPEM     | Median Rank = 7.00 | Mean AE = 0.258561
 9. SMQ_RF     | Median Rank = 7.00 | Mean AE = 0.249490
10. FSPL       | Median Rank = 9.00 | Mean AE = 0.258792
11. SMQ_BASE   | Median Rank = 10.00 | Mean AE = 0.273736
12. SMQ_ISO    | Median Rank = 11.00 | Mean AE = 0.275760

Gerando gráfico para n = 2...


  Salvo: boxplots_binario\rank_boxplot_binario_n_2.png

Ranking para n = 2
 1. FSMM       | Median Rank = 1.50 | Mean AE = 0.203605
 2. CC         | Median Rank = 4.00 | Mean AE = 0.201605
 3. FSNN       | Median Rank = 4.00 | Mean AE = 0.207913
 4. FSPL       | Median Rank = 5.00 | Mean AE = 0.205620
 5. PCC        | Median Rank = 5.00 | Mean AE = 0.212027
 6. SMQ_BASE   | Median Rank = 6.00 | Mean AE = 0.214632
 7. FSGPEM     | Median Rank = 6.00 | Mean AE = 0.215653
 8. FSGMM      | Median Rank = 7.00 | Mean AE = 0.228011
 9. EMQ        | Median Rank = 9.00 | Mean AE = 0.255708
10. SMQ_ET     | Median Rank = 10.00 | Mean AE = 0.278941
11. ACC        | Median Rank = 11.00 | Mean AE = 0.278694
12. KDEyML     | Median Rank = 11.50 | Mean AE = 0.287893
13. DyS        | Median Rank = 12.00 | Mean AE = 0.293121
14. SMQ_RF     | Median Rank = 13.00 | Mean AE = 0.303413
15. SMQ_GBR    | Median Rank = 14.00 | Mean AE = 0.309404
16. SMQ_ISO    | Median Rank = 15.00 | Mean AE = 0.329370
17. PA

  Salvo: boxplots_binario\rank_boxplot_binario_n_3.png

Ranking para n = 3
 1. FSMM       | Median Rank = 1.00 | Mean AE = 0.193613
 2. CC         | Median Rank = 4.00 | Mean AE = 0.184626
 3. FSPL       | Median Rank = 4.00 | Mean AE = 0.187519
 4. SMQ_BASE   | Median Rank = 4.50 | Mean AE = 0.186922
 5. FSNN       | Median Rank = 5.00 | Mean AE = 0.189159
 6. PCC        | Median Rank = 6.00 | Mean AE = 0.198492
 7. FSGPEM     | Median Rank = 6.50 | Mean AE = 0.199113
 8. FSGMM      | Median Rank = 8.00 | Mean AE = 0.215981
 9. EMQ        | Median Rank = 9.50 | Mean AE = 0.222506
10. DyS        | Median Rank = 10.00 | Mean AE = 0.244663
11. SMQ_ET     | Median Rank = 10.00 | Mean AE = 0.249020
12. ACC        | Median Rank = 12.00 | Mean AE = 0.258907
13. KDEyML     | Median Rank = 12.00 | Mean AE = 0.249296
14. SMQ_RF     | Median Rank = 13.50 | Mean AE = 0.269157
15. SMQ_GBR    | Median Rank = 14.00 | Mean AE = 0.264419
16. SMQ_ISO    | Median Rank = 15.50 | Mean AE = 0.285562
17. PA

  Salvo: boxplots_binario\rank_boxplot_binario_n_5.png

Ranking para n = 5
 1. CC         | Median Rank = 3.50 | Mean AE = 0.158631
 2. FSPL       | Median Rank = 4.00 | Mean AE = 0.157638
 3. SMQ_BASE   | Median Rank = 5.00 | Mean AE = 0.160019
 4. FSGPEM     | Median Rank = 6.00 | Mean AE = 0.177169
 5. FSNN       | Median Rank = 6.00 | Mean AE = 0.168214
 6. FSMM       | Median Rank = 6.50 | Mean AE = 0.185287
 7. EMQ        | Median Rank = 7.00 | Mean AE = 0.171274
 8. PCC        | Median Rank = 8.00 | Mean AE = 0.179516
 9. SMQ_GBR    | Median Rank = 10.00 | Mean AE = 0.195377
10. SMQ_ET     | Median Rank = 10.00 | Mean AE = 0.186950
11. DyS        | Median Rank = 10.50 | Mean AE = 0.194803
12. SMQ_RF     | Median Rank = 11.00 | Mean AE = 0.194555
13. KDEyML     | Median Rank = 12.00 | Mean AE = 0.200491
14. FSGMM      | Median Rank = 12.50 | Mean AE = 0.198241
15. ACC        | Median Rank = 14.00 | Mean AE = 0.214767
16. PACC       | Median Rank = 14.50 | Mean AE = 0.219125
17. S

  Salvo: boxplots_binario\rank_boxplot_binario_n_10.png

Ranking para n = 10
 1. FSPL       | Median Rank = 3.00 | Mean AE = 0.124832
 2. EMQ        | Median Rank = 3.50 | Mean AE = 0.122790
 3. CC         | Median Rank = 4.00 | Mean AE = 0.130960
 4. KDEyML     | Median Rank = 6.00 | Mean AE = 0.145494
 5. SMQ_BASE   | Median Rank = 7.00 | Mean AE = 0.135845
 6. FSNN       | Median Rank = 8.00 | Mean AE = 0.142933
 7. FSGPEM     | Median Rank = 8.00 | Mean AE = 0.152675
 8. SMQ_ET     | Median Rank = 9.00 | Mean AE = 0.151932
 9. DyS        | Median Rank = 10.00 | Mean AE = 0.154085
10. SMQ_RF     | Median Rank = 10.00 | Mean AE = 0.160140
11. FSMM       | Median Rank = 10.50 | Mean AE = 0.181599
12. SMQ_GBR    | Median Rank = 10.50 | Mean AE = 0.156371
13. PACC       | Median Rank = 11.00 | Mean AE = 0.172901
14. ACC        | Median Rank = 11.50 | Mean AE = 0.172582
15. PCC        | Median Rank = 12.50 | Mean AE = 0.156942
16. SMQ_ISO    | Median Rank = 14.00 | Mean AE = 0.170119
17.

  Salvo: boxplots_binario\rank_boxplot_binario_n_100.png

Ranking para n = 100
 1. SMQ_ET     | Median Rank = 4.00 | Mean AE = 0.053508
 2. KDEyML     | Median Rank = 4.00 | Mean AE = 0.054925
 3. SMQ_RF     | Median Rank = 4.50 | Mean AE = 0.052754
 4. EMQ        | Median Rank = 5.00 | Mean AE = 0.075131
 5. SMQ_GBR    | Median Rank = 5.00 | Mean AE = 0.053607
 6. DyS        | Median Rank = 5.50 | Mean AE = 0.056333
 7. FSPL       | Median Rank = 7.50 | Mean AE = 0.071276
 8. PACC       | Median Rank = 8.00 | Mean AE = 0.059284
 9. ACC        | Median Rank = 8.50 | Mean AE = 0.065169
10. SMQ_ISO    | Median Rank = 10.50 | Mean AE = 0.080546
11. SMQ_BASE   | Median Rank = 12.00 | Mean AE = 0.095297
12. CC         | Median Rank = 12.00 | Mean AE = 0.094581
13. FSGPEM     | Median Rank = 12.00 | Mean AE = 0.089389
14. FSNN       | Median Rank = 14.00 | Mean AE = 0.109762
15. PCC        | Median Rank = 14.50 | Mean AE = 0.120876
16. FSGMM      | Median Rank = 16.00 | Mean AE = 0.182191
17

  Salvo: boxplots_binario\rank_boxplot_binario_n_full.png

Ranking para n = full
 1. SMQ_ET     | Median Rank = 4.00 | Mean AE = 0.041382
 2. SMQ_GBR    | Median Rank = 4.00 | Mean AE = 0.040222
 3. KDEyML     | Median Rank = 4.00 | Mean AE = 0.042754
 4. DyS        | Median Rank = 5.00 | Mean AE = 0.043965
 5. SMQ_RF     | Median Rank = 5.00 | Mean AE = 0.041047
 6. PACC       | Median Rank = 6.00 | Mean AE = 0.046127
 7. ACC        | Median Rank = 7.00 | Mean AE = 0.053496
 8. SMQ_ISO    | Median Rank = 9.00 | Mean AE = 0.070357
 9. CC         | Median Rank = 10.00 | Mean AE = 0.091047
10. FSGPEM     | Median Rank = 10.50 | Mean AE = 0.082448
11. FSPL       | Median Rank = 11.00 | Mean AE = 0.159616
12. FSNN       | Median Rank = 13.00 | Mean AE = 0.125504
13. PCC        | Median Rank = 13.00 | Mean AE = 0.115527
14. SMQ_BASE   | Median Rank = 13.00 | Mean AE = 0.121994
15. EMQ        | Median Rank = 13.00 | Mean AE = 0.171263
16. FSMM       | Median Rank = 16.00 | Mean AE = 0.253283

In [ ]:
testar outras tecnicas de calibração multiclasse no smq iso tipo temperature scaling e dirthclet 
hibrido vs outros metodos de oversampling